In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df_matches = pd.read_csv('data/matches.csv')
print("First 5 rows of the DataFrame:")
display(df_matches.head())

print(f"Shape of the DataFrame: {df_matches.shape}")
print("\nColumn names:")
for col in df_matches.columns:
    print(col)

unique_seasons = df_matches['season'].nunique()
print(f"Number of unique seasons in the dataset: {unique_seasons}")

missing_values = df_matches.isnull().sum()
missing_values = missing_values[missing_values > 0]
if not missing_values.empty:
    print("Columns with missing values:")
    display(missing_values)
else:
    print("No missing values found in any column.")

matches_per_season = df_matches['season'].value_counts().sort_index()
plt.figure(figsize=(12, 6))
sns.barplot(x=matches_per_season.index, y=matches_per_season.values, palette='viridis')
plt.title('Number of Matches Played Per Season')
plt.xlabel('Season')
plt.ylabel('Number of Matches')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
toss_winner_match_winner = df_matches[df_matches['toss_winner'] == df_matches['winner']]
percentage_won_by_toss_winner = (len(toss_winner_match_winner) / len(df_matches)) * 100
print(f"Percentage of matches won by the team that also won the toss: {percentage_won_by_toss_winner:.2f}%")

In [ ]:
toss_decision_wins = df_matches.groupby(['season', 'toss_decision', 'winner']).size().unstack(fill_value=0)
def calculate_win_percentage(row):
    toss_winner = row.name[2] 
    total_matches_with_decision = row.sum()
    if toss_winner in row.index:
        return (row[toss_winner] / total_matches_with_decision) * 100
    return 0
win_percentages = []
for index, row in df_matches.iterrows():
    toss_winner = row['toss_winner']
    toss_decision = row['toss_decision']
    match_winner = row['winner']
    if match_winner == toss_winner:
        win_percentages.append({
            'season': row['season'],
            'toss_decision': toss_decision,
            'won_match_after_toss_decision': 1  
        })
    else:
        win_percentages.append({
            'season': row['season'],
            'toss_decision': toss_decision,
            'won_match_after_toss_decision': 0  
        })
df_win_percentages = pd.DataFrame(win_percentages)
win_percentage_by_decision_season = df_win_percentages.groupby(['season', 'toss_decision'])['won_match_after_toss_decision'].mean().unstack(fill_value=0) * 100
display(win_percentage_by_decision_season.head())

In [ ]:
plot_data = win_percentage_by_decision_season.stack().reset_index(name='win_percentage')
plot_data.rename(columns={'level_1': 'toss_decision'}, inplace=True)
plt.figure(figsize=(15, 8))
sns.barplot(x='season', y='win_percentage', hue='toss_decision', data=plot_data, palette='coolwarm')
plt.title('Win Percentage by Toss Decision Across Seasons')
plt.xlabel('Season')
plt.ylabel('Win Percentage (%)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Toss Decision')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()